# Skin atlas — low-resolution broad annotation

Loads the **skin-only** cells from the joint atlas (`compartment == "Skin"`, ~866k cells, 6 studies)
with their cached **MRVI** latents (`joint_X_mrvi_u_skin.npy` = sample-unaware integration view).
Computes a UMAP, **low-resolution Leiden** (broad lineages), and a marker **dot plot** — using the
Li2024 `cell_type` labels as the naming scaffold (the other skin cohorts are `Unknown`).

**HEAVY cells run on the GPU kernel (`neural_nmf_env`).** UMAP coords + Leiden labels are cached so
re-runs reload instead of recomputing.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc


def _resolve_nb_dir():
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(start)


NB_DIR = _resolve_nb_dir()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1
OUT = NB_DIR / "data" / "atlas_joint"

ANNOT = OUT / "joint_annotated.h5ad"
MRVI_U = OUT / "joint_X_mrvi_u_skin.npy"   # sample-unaware (integration view), 10-d
MRVI_Z = OUT / "joint_X_mrvi_skin.npy"     # sample-aware, 30-d
UMAP_NPY = OUT / "joint_umap_skin_u.npy"
LEIDEN_CSV = OUT / "skin_leiden_broad.csv"
print("NB_DIR:", NB_DIR)

## 1 — Load the skin subset + MRVI latents

`compartment == "Skin"` row order matches the cached `_skin` latents (the MRVI job was run on this
exact slice, see nb 10 step 7b). **HEAVY** (reads the 1.34M-cell atlas) — GPU kernel.

In [ ]:
adata = sc.read_h5ad(ANNOT)
ad = adata[adata.obs["compartment"] == "Skin"].copy()
del adata

ad.obsm["X_mrvi_u"] = np.load(MRVI_U)   # integration view (drives neighbors/UMAP/Leiden)
ad.obsm["X_mrvi_z"] = np.load(MRVI_Z)   # sample-aware (kept for later substructure work)
assert ad.n_obs == ad.obsm["X_mrvi_u"].shape[0], "skin slice / latent row mismatch"
print(ad)
print("X_mrvi_u", ad.obsm["X_mrvi_u"].shape, "| X_mrvi_z", ad.obsm["X_mrvi_z"].shape)

## 2 — Existing labels (naming scaffold)

Which lineages already carry Li2024 `cell_type` names vs `Unknown` (the rows we annotate here).

In [ ]:
print("--- cell_type (skin) ---")
print(ad.obs["cell_type"].value_counts().head(25).to_string())
print("\n--- cell_type x study (Unknown = unlabelled cohorts) ---")
print(pd.crosstab(ad.obs["cell_type"], ad.obs["study"]).to_string())

## 3 — Log-normalize for marker viz

Atlas `X` is raw counts. Keep counts in a layer; log-normalize `X` for the dot plot / marker overlays.

In [ ]:
ad.layers["counts"] = ad.X.copy()
sc.pp.normalize_total(ad, target_sum=1e4)
sc.pp.log1p(ad)
print("X is log1p-normalized:", float(ad.X.max()) < 20)

## 4 — Neighbors + UMAP on `X_mrvi_u` (HEAVY, cached)

Coords cached to `joint_umap_skin_u.npy`. The neighbor graph is still rebuilt on reload (needed for
Leiden) but UMAP layout — the slow part — is skipped.

In [ ]:
np.random.seed(SEED)
sc.pp.neighbors(ad, use_rep="X_mrvi_u", random_state=SEED)   # graph needed for both UMAP + Leiden

if UMAP_NPY.exists():
    ad.obsm["X_umap"] = np.load(UMAP_NPY)
    print("loaded cached UMAP", ad.obsm["X_umap"].shape)
else:
    sc.tl.umap(ad, random_state=SEED)
    np.save(UMAP_NPY, ad.obsm["X_umap"])
    print("computed + cached UMAP ->", UMAP_NPY)

## 5 — Low-resolution Leiden (broad lineages, cached)

`resolution=0.2` → a handful of clusters for the major compartments. Labels cached to
`skin_leiden_broad.csv` (`cell_id → leiden_broad`).

In [ ]:
LEIDEN_RES = 0.7
FORCE_LEIDEN = True   # True -> recompute (+ overwrite cache) even if skin_leiden_broad.csv exists

if LEIDEN_CSV.exists() and not FORCE_LEIDEN:
    lab = pd.read_csv(LEIDEN_CSV, index_col="cell_id")["leiden_broad"].astype(str)
    ad.obs["leiden_broad"] = lab.reindex(ad.obs["cell_id"].astype(str)).to_numpy()
    ad.obs["leiden_broad"] = ad.obs["leiden_broad"].astype("category")
    print("loaded cached Leiden labels (set FORCE_LEIDEN=True to recompute)")
else:
    sc.tl.leiden(ad, resolution=LEIDEN_RES, random_state=SEED,
                 key_added="leiden_broad", flavor="igraph", n_iterations=2, directed=False)
    pd.DataFrame({"cell_id": ad.obs["cell_id"].astype(str).to_numpy(),
                  "leiden_broad": ad.obs["leiden_broad"].astype(str).to_numpy()}) \
        .to_csv(LEIDEN_CSV, index=False)
    print(f"computed Leiden (res={LEIDEN_RES}) + cached ->", LEIDEN_CSV)

print("clusters:", ad.obs["leiden_broad"].value_counts().sort_index().to_dict())

## 6 — UMAP overview

In [ ]:
sc.pl.umap(
    ad,
    color=["leiden_broad", "cell_type", "study", "disease"],
    ncols=2, frameon=False, size=2, wspace=0.25,
)

## 7 — Marker dot plot (broad lineages)

Each Leiden cluster should light up one marker block. `standard_scale="var"` scales each gene 0–1
across clusters. Genes absent from the 49,683-gene outer-join are dropped first.

In [ ]:
markers = {
    "T": ["CD3D", "CD3E", "TRAC"],
    "CD4": ["CD4", "IL7R"],
    "CD8/NK": ["CD8A", "GZMB", "NKG7", "GNLY"],
    "Treg": ["FOXP3", "CTLA4"],
    "B": ["MS4A1", "CD79A", "CD19"],
    "Plasma": ["MZB1", "JCHAIN", "IGHG1"],
    "Myeloid/DC": ["LYZ", "CD68", "ITGAX", "CD14"],
    "Mast": ["TPSAB1", "CPA3", "KIT"],
    "Keratinocyte": ["KRT14", "KRT5", "KRT1"],
    "Fibroblast": ["COL1A1", "DCN", "LUM"],
    "Vascular endo": ["PECAM1", "VWF", "CLDN5"],
    "Lymphatic endo": ["LYVE1", "PROX1"],
    "Melanocyte": ["MLANA", "PMEL", "TYR"],
}
genes_present = set(ad.var_names)
missing = sorted({g for v in markers.values() for g in v} - genes_present)
if missing:
    print("genes absent from var_names (dropped):", missing)
markers = {k: [g for g in v if g in genes_present] for k, v in markers.items()}
markers = {k: v for k, v in markers.items() if v}   # drop empty groups

sc.pl.dotplot(ad, markers, groupby="leiden_broad", standard_scale="var", dendrogram=True)

## 8 — Map clusters → cell type

Fill `cluster2ct` from the dot plot (§7) + the Li2024 labels (§2). Run the helper print first if
clusters changed (new `LEIDEN_RES`) to get a fresh key template.

In [ ]:
# template (re-run if clusters changed): print({c: "" for c in ad.obs["leiden_broad"].cat.categories})

cluster2ct = {
    "0": "Ketratinocyte",
    "1": "Ketratinocyte",
    "2": "Ketratinocyte",
    "3": "Ketratinocyte",
    "4": "T",
    "5": "T",
    "6": "T",
    "7": "Myeloid",
    "8": "Myeloid",
    "9": "Ketratinocyte",
    "10": "T",
    "11": "T",
    "12": "T",
    "13": "B",
    "14": "Myeloid",
    "15": "Plasma",
    "16": "Plasma",
    "17": "Vascular",
    "18": "Fibroblast",
    "19": "Melanocyte",
    "20": "Fibroblast",
    "21": "Plasma",
    
}
assert set(cluster2ct) == set(ad.obs["leiden_broad"].cat.categories), \
    "keys must match leiden_broad clusters: " + str(sorted(ad.obs["leiden_broad"].cat.categories))

ad.obs["cell_type_broad"] = ad.obs["leiden_broad"].map(cluster2ct).astype("category")
print(ad.obs["cell_type_broad"].value_counts(dropna=False).to_string())

## 9 — Compare to the previous (Li2024) annotation

Where Li2024 `cell_type` exists (`!= Unknown`), cross-tab it against the new `cell_type_broad` to
check each broad label captures a coherent set of known types. Skipped if `cluster2ct` is unfilled.

In [ ]:
if "cell_type_broad" not in ad.obs or ad.obs["cell_type_broad"].astype(str).str.len().eq(0).all():
    print("cell_type_broad unfilled — complete cluster2ct in §8 first.")
else:
    known = ad.obs[ad.obs["cell_type"] != "Unknown"].copy()
    print(f"Li2024-labelled skin cells: {len(known):,} / {ad.n_obs:,}")

    cmp = pd.crosstab(known["cell_type_broad"], known["cell_type"])
    cmp = cmp.loc[:, cmp.sum() > 0]
    frac = cmp.div(cmp.sum(axis=1).clip(lower=1), axis=0).round(2)
    with pd.option_context("display.max_columns", None, "display.width", 240):
        print("\nrow-normalized cell_type_broad x Li2024 cell_type:")
        print(frac.to_string())
        print("\ndominant Li2024 label per broad annotation (purity):")
        summary = pd.DataFrame({"top_li2024": frac.idxmax(axis=1),
                                "purity": frac.max(axis=1),
                                "n_known": cmp.sum(axis=1)})
        print(summary.to_string())

In [ ]:
sc.pl.umap(
    ad,
    color=["leiden_broad", "cell_type_broad", "study", "disease"],
    ncols=2, frameon=False, size=2, wspace=0.25,
)

# T-cell sub-clustering

Subset to the broad **T** compartment (`cell_type_broad == "T"`, ~402k cells), recompute
neighbors/UMAP/Leiden on `X_mrvi_u` for this slice only, dot-plot T/NK markers, annotate subtypes,
and cache the complete T-cell `AnnData`.

## 10 — Subset to T cells

In [ ]:
# T-cell sub-cluster cache paths
T_UMAP_NPY = OUT / "skin_T_umap_u.npy"
T_LEIDEN_CSV = OUT / "skin_T_leiden.csv"
T_ANNOT = OUT / "skin_T_annotated.h5ad"

adt = ad[ad.obs["cell_type_broad"] == "T"].copy()   # carries X (log1p), counts layer, X_mrvi_u/z
print(adt)
print("T cells:", adt.n_obs, "| X_mrvi_u", adt.obsm["X_mrvi_u"].shape)

## 11 — Neighbors + UMAP on the T subset (HEAVY, cached)

Re-run neighbors/UMAP on `X_mrvi_u` for T cells only so the embedding resolves T substructure
instead of the global-lineage layout. Coords cached to `skin_T_umap_u.npy`.

In [ ]:
np.random.seed(SEED)
sc.pp.neighbors(adt, use_rep="X_mrvi_u", random_state=SEED)   # rebuild graph on T-only subset

if T_UMAP_NPY.exists():
    adt.obsm["X_umap"] = np.load(T_UMAP_NPY)
    print("loaded cached T UMAP", adt.obsm["X_umap"].shape)
else:
    sc.tl.umap(adt, random_state=SEED)
    np.save(T_UMAP_NPY, adt.obsm["X_umap"])
    print("computed + cached T UMAP ->", T_UMAP_NPY)

## 12 — Leiden on T cells (cached)

`resolution=1.0` → finer T/NK subtypes. Labels cached to `skin_T_leiden.csv` (`cell_id → leiden_T`).

In [ ]:
T_LEIDEN_RES = 1.0
FORCE_T_LEIDEN = True

if T_LEIDEN_CSV.exists() and not FORCE_T_LEIDEN:
    lab = pd.read_csv(T_LEIDEN_CSV, index_col="cell_id")["leiden_T"].astype(str)
    adt.obs["leiden_T"] = lab.reindex(adt.obs["cell_id"].astype(str)).to_numpy()
    adt.obs["leiden_T"] = adt.obs["leiden_T"].astype("category")
    print("loaded cached T Leiden labels (set FORCE_T_LEIDEN=True to recompute)")
else:
    sc.tl.leiden(adt, resolution=T_LEIDEN_RES, random_state=SEED,
                 key_added="leiden_T", flavor="igraph", n_iterations=2, directed=False)
    pd.DataFrame({"cell_id": adt.obs["cell_id"].astype(str).to_numpy(),
                  "leiden_T": adt.obs["leiden_T"].astype(str).to_numpy()}) \
        .to_csv(T_LEIDEN_CSV, index=False)
    print(f"computed T Leiden (res={T_LEIDEN_RES}) + cached ->", T_LEIDEN_CSV)

print("clusters:", adt.obs["leiden_T"].value_counts().sort_index().to_dict())

## 13 — T-cell marker dot plot

Granular T/NK markers per `leiden_T` cluster. `standard_scale="var"` scales each gene 0–1 across
clusters; genes absent from the outer-join are dropped first.

In [ ]:
markers_T = {
    "T core": ["CD3D", "CD3E", "TRAC"],
    "CD4": ["CD4", "IL7R"],
    "CD8": ["CD8A", "CD8B"],
    "Naive/mem": ["CCR7", "SELL", "TCF7"],
    "Cytotoxic": ["GZMK", "GZMB", "PRF1", "NKG7", "GNLY"],
    "Treg": ["FOXP3", "CTLA4", "IL2RA"],
    "Th17/Tc17": ["RORC", "IL17A", "CCR6"],
    "Exhaustion": ["PDCD1", "TOX", "LAG3", "TIGIT", "HAVCR2"],
    "NK": ["NCAM1", "KLRD1", "KLRF1"],
    "ILC": ["KIT", "GATA3"],
    "Prolif": ["MKI67", "TOP2A"],
}
genes_present = set(adt.var_names)
missing = sorted({g for v in markers_T.values() for g in v} - genes_present)
if missing:
    print("genes absent from var_names (dropped):", missing)
markers_T = {k: [g for g in v if g in genes_present] for k, v in markers_T.items()}
markers_T = {k: v for k, v in markers_T.items() if v}

sc.pl.dotplot(adt, markers_T, groupby="leiden_T", standard_scale="var", dendrogram=True)

## 13.b — Map T clusters → subtype

Fill `cluster2ct_T` from the §13 dot plot + Li2024 `cell_type` (CD4/Th, CD8/Tc, Treg, NK/ILC,
cytotoxic, proliferating, malignant `tumor_cell`). Run with empty values first to print the template.

In [ ]:
# template (re-run if clusters changed): print({c: "" for c in adt.obs["leiden_T"].cat.categories})

cluster2ct_T = {
    "0": "CD4_mem",
    "1": "CD4_mem",
    "2": "CD4_Th2",
    "3": "CD4_Th2",
    "4": "CD4_Th2",
    "5": "CD4_Th2",
    "6": "CD4_Th2",
    "7": "CD4_Th2",
    "8": "CD8_em",
    "9": "CD8_effector",
    "10": "CD4_Treg",
    "11": "CD4_Cm",
    "12": "CD4_Cm",
    "13": "UNK",
    "14": "CD8_effector",
    "15": "CD4_cytotoxic",
 
}
assert set(cluster2ct_T) == set(adt.obs["leiden_T"].cat.categories), \
    "keys must match leiden_T clusters: " + str(sorted(adt.obs["leiden_T"].cat.categories))

if any(v == "" for v in cluster2ct_T.values()):
    print("cluster2ct_T unfilled — read the §13 dot plot, then map each leiden_T cluster.")
    print("clusters:", adt.obs["leiden_T"].value_counts().sort_index().to_dict())
else:
    adt.obs["cell_type_T"] = adt.obs["leiden_T"].map(cluster2ct_T).astype("category")
    print(adt.obs["cell_type_T"].value_counts(dropna=False).to_string())

    # sanity vs Li2024 labels
    known = adt.obs[adt.obs["cell_type"] != "Unknown"].copy()
    cmp = pd.crosstab(known["cell_type_T"], known["cell_type"])
    frac = cmp.div(cmp.sum(axis=1).clip(lower=1), axis=0).round(2)
    summary = pd.DataFrame({"top_li2024": frac.idxmax(axis=1),
                            "purity": frac.max(axis=1),
                            "n_known": cmp.sum(axis=1)})
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print("\ndominant Li2024 label per cell_type_T (purity):")
        print(summary.to_string())

## 13.c — T-cell UMAP overview

T-only UMAP coloured by `leiden_T`, the new `cell_type_T`, `study` and `sample_id` — check the
subtypes are coherent and not driven by a single cohort/sample.

In [ ]:
colors = ["leiden_T"]
if "cell_type_T" in adt.obs:
    colors.append("cell_type_T")
colors += ["study", "sample_id"]

sc.pl.umap(
    adt,
    color=colors,
    ncols=2, frameon=False, size=2, wspace=0.25,
    legend_loc="on data" if "cell_type_T" in adt.obs else "right margin",
)

In [ ]:
sc.pl.umap(
    adt,
    color=["GATA3","CCR4","CXCL13","TCF7","SELL","GZMB","GNLY","KIR3DL2"],
    ncols=2, frameon=False, size=2, wspace=0.25,
    legend_loc="on data" if "cell_type_T" in adt.obs else "right margin",
)

## 14 — Cache the complete T-cell adata

Full T-cell `AnnData` (log1p `X`, `counts` layer, MRVI latents, T-UMAP, `leiden_T`, `cell_type_T`)
written to `skin_T_annotated.h5ad` for downstream substructure work.

In [ ]:
adt.write_h5ad(T_ANNOT)
print("wrote complete T-cell adata ->", T_ANNOT)
print(adt)

In [ ]:
adt.obs.study.value_counts().index.tolist()